In [ ]:
"""
Diagnostic financier d'un grand livre comptable multi-devises
================================================================

Contexte
--------
Analyse d'un export de grand livre comptable (General Ledger) de 2000 écritures,
couvrant 3 exercices (2023-2025), 6 départements et 5 devises (USD, GBP, EUR, AUD, CAD).

Objectifs
---------
1. Contrôler la qualité et la cohérence du grand livre
2. Valider la fiabilité de la conversion multi-devises
3. Construire un compte de résultat consolidé en USD
4. Analyser la répartition des charges par département
5. Investiguer une variation significative détectée sur l'exercice 2025

Source des données : General-Ledger.xlsx (Jrtoby/General-Ledger-Financial-Analysis, GitHub)
Auteur : N'KOUEI N'tcha Gildas
"""

In [ ]:
import pandas as pd

In [ ]:
URL = "https://raw.githubusercontent.com/Jrtoby/General-Ledger-Financial-Analysis/main/General-Ledger.xlsx"

In [ ]:
# ══════════════════════════════════════════════════════════════════
# A. CHARGEMENT ET PRISE EN MAIN
# ══════════════════════════════════════════════════════════════════

gl = pd.read_excel(URL, sheet_name="GL", usecols=[
    "GLID", "TxnDate", "Year", "Month", "AccountNumber", "AccountName",
    "Debit", "Credit", "Dept", "CostCenter", "Description", "Currency",
    "NetAmount", "RateToUSD", "Debit_USD", "Credit_USD", "NetAmountUSD", "AccountType"
])
taux_change = pd.read_excel(URL, sheet_name="ExchangeRate")

print("=" * 60)
print("A. PRISE EN MAIN")
print("=" * 60)
print(f"Dimensions du grand livre : {gl.shape}")
print(f"\nTypes de données :\n{gl.dtypes}")
print(f"\nDépartements : {gl['Dept'].unique()}")
print(f"Comptes : {gl['AccountName'].unique()}")
print(f"Types de compte : {gl['AccountType'].unique()}")
print(f"Devises : {gl['Currency'].unique()}")
print(f"\nValeurs manquantes :\n{gl.isna().sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# B. CONTRÔLE QUALITÉ — ÉQUILIBRE DU GRAND LIVRE
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("B. CONTRÔLE QUALITÉ — ÉQUILIBRE DÉBIT/CRÉDIT")
print("=" * 60)

type_group = gl.groupby("AccountType")[["Debit", "Credit"]].sum()
print(type_group)

# Le grand livre ne couvre que les comptes de résultat (Revenue, COGS, Expense),
# sans les comptes de bilan (trésorerie, tiers) : l'écart Débit/Crédit observé
# correspond donc au résultat avant impôt, pas à une erreur de saisie.
resultat_avant_impot_local = (
    type_group.loc["Revenue", "Credit"]
    - (type_group.loc["COGS", "Debit"] + type_group.loc["Expense", "Debit"])
)
print(f"\nRésultat avant impôt (devise locale) : {resultat_avant_impot_local:,.2f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# C. VALIDATION DE LA CONVERSION MULTI-DEVISES
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("C. VALIDATION DE LA CONVERSION MULTI-DEVISES")
print("=" * 60)

devises = gl.merge(taux_change, on="Currency", how="left")
devises["Debit2USD"] = devises["Debit"] * devises["Rate2USD"]
devises["Credit2USD"] = devises["Credit"] * devises["Rate2USD"]
devises["ecart_debit"] = devises["Debit2USD"] - devises["Debit_USD"]
devises["ecart_credit"] = devises["Credit2USD"] - devises["Credit_USD"]

print(f"Écarts de conversion (débit) : {devises['ecart_debit'].unique()}")
print(f"Écarts de conversion (crédit) : {devises['ecart_credit'].unique()}")
print("→ Aucun écart détecté : la conversion USD fournie est fiable.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# D. CONSTRUCTION DU COMPTE DE RÉSULTAT (P&L), EN USD
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("D. COMPTE DE RÉSULTAT CONSOLIDÉ (USD)")
print("=" * 60)

# Table de mapping : compte comptable -> poste de reporting, avec sens normalisé
mapping_poste = {
    "Revenue": "Chiffre d'affaires",
    "COGS": "Coût des ventes",
    "Expense": "Charges opérationnelles",
}
sens_normal = {
    "Chiffre d'affaires": -1,   # sens normal au crédit
    "Coût des ventes": 1,        # sens normal au débit
    "Charges opérationnelles": 1,
}

pnl = devises[["AccountType", "NetAmountUSD"]].copy()
pnl["poste_reporting"] = pnl["AccountType"].map(mapping_poste)
pnl["sens"] = pnl["poste_reporting"].map(sens_normal)
pnl["montant"] = pnl["NetAmountUSD"] * pnl["sens"]

In [ ]:
compte_resultat = pnl.groupby("poste_reporting")["montant"].sum()

ca = compte_resultat.loc["Chiffre d'affaires"]
cogs = compte_resultat.loc["Coût des ventes"]
opex = compte_resultat.loc["Charges opérationnelles"]

marge_brute = ca - cogs
taux_marge_brute = marge_brute / ca
resultat_avant_impot = marge_brute - opex
TAUX_IS = 0.30  # hypothèse 
impot = resultat_avant_impot * TAUX_IS
resultat_net = resultat_avant_impot - impot

print(f"Chiffre d'affaires        : {ca:>15,.0f} USD")
print(f"Coût des ventes (COGS)     : {cogs:>15,.0f} USD")
print(f"Marge brute                : {marge_brute:>15,.0f} USD  ({taux_marge_brute:.1%})")
print(f"Charges opérationnelles    : {opex:>15,.0f} USD")
print(f"Résultat avant impôt       : {resultat_avant_impot:>15,.0f} USD")
print(f"Impôt (hyp. {TAUX_IS:.0%})           : {impot:>15,.0f} USD")
print(f"Résultat net               : {resultat_net:>15,.0f} USD")

In [ ]:
# Tableau final pour Excel
compte_resultat_final = pd.DataFrame({
    "Poste": ["Chiffre d'affaires", "Coût des ventes", "Marge brute", 
              "Charges opérationnelles", "Résultat avant impôt", 
              f"Impôt ({TAUX_IS:.0%})", "Résultat net"],
    "Montant_USD": [ca, -cogs, marge_brute, -opex, resultat_avant_impot, -impot, resultat_net]
})

In [ ]:
# ══════════════════════════════════════════════════════════════════
# E. ANALYSE PAR DÉPARTEMENT ET CENTRE DE COÛT
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("E. ANALYSE PAR DÉPARTEMENT")
print("=" * 60)

# Convention : le COGS est inclus dans les charges par département,
# car directement rattaché à l'activité opérationnelle de chaque département.
par_departement = devises.groupby("Dept")["Debit_USD"].sum().sort_values(ascending=False)
print(f"Charges totales par département (COGS inclus) :\n{par_departement}")

croisement_dept = devises.pivot_table(
    index="Dept", values="Debit_USD", columns="AccountType", aggfunc="sum"
).fillna(0)
print(f"\nDétail par département et type de compte :\n{croisement_dept}")

nb_ecritures_par_compte = gl["AccountNumber"].value_counts()
print(f"\nNombre d'écritures par compte :\n{nb_ecritures_par_compte}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# F. INVESTIGATION — VARIATION DE CHIFFRE D'AFFAIRES SUR 2025
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("F. INVESTIGATION — EXERCICE 2025")
print("=" * 60)

devises["TxnDate"] = pd.to_datetime(devises["TxnDate"])
group = devises.groupby(["Year", "Month", "AccountName"])["NetAmountUSD"].sum().unstack(fill_value=0)
group["Chiffre_affaires"] = (group["Online Sales"] + group["Sales Revenue"]) * -1
group["Charges_totales"] = group["COGS"] + group["Payroll Expense"] + group["Travel Expense"]
group["resultat_net"] = (group["Chiffre_affaires"] - group["Charges_totales"]) * (1 - TAUX_IS)

annuel = group.groupby("Year")[["Chiffre_affaires", "resultat_net"]].sum()
annuel["YoY_CA_%"] = annuel["Chiffre_affaires"].pct_change() * 100
print(f"CA et résultat net par année :\n{annuel}")

In [ ]:
# Vérification avant de conclure à une baisse d'activité : l'exercice 2025
# couvre-t-il bien 12 mois, ou est-il en cours au moment de l'extraction ?
ca_2025 = group.loc[2025, "Chiffre_affaires"]
ca_2023_2024 = group.loc[[2023, 2024], "Chiffre_affaires"]
print(f"\nMois couverts en 2025 : {len(ca_2025)} / 12")
print(f"CA moyen mensuel 2025 : {ca_2025.mean():,.0f} USD")
print(f"CA moyen mensuel 2023-2024 : {ca_2023_2024.mean():,.0f} USD")
print(f"Écart de rythme mensuel : {(ca_2025.mean() / ca_2023_2024.mean() - 1):.1%}")
print("→ 2025 est un exercice incomplet (données arrêtées à fin juin), pas une baisse d'activité :")
print("  le rythme mensuel moyen reste proche de l'historique.")

# Saisonnalité, en excluant l'exercice 2025 incomplet
group_sans_2025 = group[group.index.get_level_values("Year") != 2025]
mensuel = group_sans_2025.groupby("Month")[["Chiffre_affaires", "resultat_net"]].sum()

MOIS_ORDRE = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
              "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
mensuel.index = mensuel.index.map(MOIS_ORDRE)
mensuel = mensuel.sort_index()
mensuel["variation_%"] = mensuel["Chiffre_affaires"].pct_change() * 100

print(f"\nSaisonnalité du CA (2023-2024 uniquement) :\n{mensuel}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXPORT EXCEL
# ══════════════════════════════════════════════════════════════════
with pd.ExcelWriter("Compte_Resultat_GrandLivre_USD.xlsx") as writer:
    compte_resultat_final.to_excel(writer, sheet_name="P&L_Consolide", index=False)
    annuel.to_excel(writer, sheet_name="P&L_Annuel")
    mensuel.to_excel(writer, sheet_name="Saisonnalite")
    par_departement.to_excel(writer, sheet_name="Par_Dept")

print("Fichier créé : Compte_Resultat_GrandLivre_USD.xlsx")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CONCLUSIONS ET RECOMMANDATIONS
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("SYNTHÈSE")
print("=" * 60)
print("""
1. L'écart Débit/Crédit du grand livre s'explique par le périmètre du dataset
   (comptes de résultat uniquement) et correspond au résultat avant impôt.
2. La conversion multi-devises fournie est fiable (0 écart détecté).
3. Résultat net consolidé : voir section D (hypothèse IS 30% à valider client).
4. Le département HR concentre le plus de charges ; Operations le moins.
5. L'exercice 2025 est incomplet — ne pas le comparer aux exercices clos
   sans ajuster le périmètre de comparaison (même nombre de mois).
6. Recommandation : plafonner l'analyse de tendance annuelle 2025 à une
   comparaison à périmètre de mois identique, et documenter explicitement
   le taux d'IS utilisé dans tout reporting basé sur ce résultat net.
""")